# Luma Phase 2 Module Implementation
This notebook contain the implementation of phase 2 of Luma Geospatial Engine

## Prerequisite
Earth engine initialization using service account

In [ ]:
import ee 
import luma_ge
#Manuall Authentication, use personal account (uncomment if needed)
#luma_ge.authenticate_manually()
#Autheticate using service account (json file)
#use relevant file in auth folder
service_account_path = '../auth/python-ee-487304-3ab71cc11a87.json'
success = luma_ge.initialize_with_service_account(service_account_path)
if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")

## Retrieve AOI from Earth Engine Asset Manager

In [ ]:
from luma_ge.data_acquisition import GEE_Asset_Manager
#Initialize the Asset Manager from Earth Engine
asset = GEE_Asset_Manager()
if asset.load_asset():
    regency_names = asset.get_regency_names()
    if regency_names:
        print(f"✓ Found {len(regency_names)} regencies")
        #use city name
        selected_regency = "Kota Bandung" 
        #Verify the city name exists in the regency list
        if selected_regency in regency_names:
            print(f"\n✓ Selected regency: {selected_regency}")
        else:
            print(f"\n✗ Regency '{selected_regency}' not found")
            print(f"\nAvailable regencies:")
            for i, name in enumerate(regency_names, 1):
                print(f"  {i}. {name}")
            selected_regency = None
    else:
        print("✗ No regency names found")
        selected_regency = None
else:
    print("✗ Failed to load asset")
    selected_regency = None
#Retrieve AOI geometry for the selected regency
if selected_regency:
    aoi = asset.get_regency_geometry(selected_regency)
    if aoi:
        print(f"✓ Successfully retrieved geometry for: {selected_regency}")
    else:
        print(f"✗ Failed to retrieve geometry for: {selected_regency}")
        aoi = None
else:
    print("✗ No regency selected")
    aoi = None

## New Feature: Sentinel-2 Data Retrieval
Note: Sentinel 2 data has varying spatial resolution. Use Sharpen=True, if spatial resolution harmonization is desired.
The approach downscale the 20m bands into 10m band, while removing the 60m band

In [3]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image
import geemap
aoi = geemap.shp_to_ee('../data/area_of_interest.shp')
#initialize class
optical_reflectance = Reflectance_Data()
composite = final_Image()
#define the temporal range
start = '2024-01-01'
end = '2024-12-30'
#retrieve the sentinel 2 data image collection
s2_data, metas2 = optical_reflectance.get_s2_optical_data(aoi, start, end, cloud_cover=20, compute_detailed_stats=False)
#final output: composite image. image sharpening is optional, if not applied, use sharpen=False
median_s2 = composite.get_temporal_composite(s2_data, aoi, calculate_coverage=False, coverage_scale=10, sharpen=True)
#Visualization for 10m band
vis_10m = {'min': 0,'max': 4000,'gamma': [0.95, 1.1, 1],'bands':['RED', 'GREEN', 'BLUE']}
#Visualization for 20m band
res_20m = {'min': 0,'max': 4000,'gamma': [0.95, 1.1, 1],'bands':['RED_EDGE1', 'RED_EDGE4', 'SWIR1']}
Map = geemap.Map()
Map.addLayer(median_s2, vis_10m, 'Sentinel-2 Composite')
Map.centerObject(aoi, 10)
Map

2026-06-05 10:28:09,117 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-06-05 10:28:09,118 - final_Image - INFO - final_Image creation initialized.
2026-06-05 10:28:09,118 - Reflectance_Data - INFO - Starting data fetch for Sentinel-2 Level-2A Surface Reflectance (Harmonized)
2026-06-05 10:28:09,118 - Reflectance_Data - INFO - Date range: 2024-01-01 to 2024-12-30
2026-06-05 10:28:09,119 - Reflectance_Data - INFO - Cloud cover threshold (image-level): 20%
2026-06-05 10:28:09,119 - Reflectance_Data - INFO - Cloud Score+ pixel threshold: 0.6
2026-06-05 10:28:09,120 - Reflectance_Data - INFO - Detailed statistics will not be computed
2026-06-05 10:28:09,121 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-06-05 10:28:09,123 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-06-05 10:28:10,663 - final_Image - INFO - Creating median composite from 4 images
2026-06-05 10:28:10,664 - final_Image - IN

Map(center=[-2.010348583322315, 103.85096772818214], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
export_band = sharpen_s2.select('RED_EDGE1','RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2')
export_task = ee.batch.Export.image.toDrive(
     image=export_band,
     description='S2_SharpenLuma_v1',
     folder='Earth Engine',
     fileNamePrefix='S2_SharpenLuma_v1',
     scale=10,
     region=aoi,  # or aoi.geometry()
     maxPixels=1e13
 )
export_task.start()
import time

while export_task.active():
     print('Exporting... (status: {})'.format(export_task.status()['state']))
     time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))